## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: July 5, 2025

# Purpose: This notebook performs various tasks labeled "Exercises" for Week 5/6 of the class. Data must be present within the folder that this notebook is run in.
# This week focuses on Data Cleaning and Transforming

In [67]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# This data had a unicode error, so we handle it with a try-except block.
try:
	candy_df = pd.read_csv('candyhierarchy2017.csv')
except UnicodeDecodeError:
	candy_df = pd.read_csv('candyhierarchy2017.csv', encoding='latin1')

In [68]:
# Display summary statistics
print(candy_df.describe(include='all'))

# Print the first few rows
print(candy_df.head())

# Print column names and their data types
print(candy_df.dtypes)

         Internal ID Q1: GOING OUT? Q2: GENDER Q3: AGE Q4: COUNTRY  \
count   2.460000e+03           2350       2419    2376        2396   
unique           NaN              2          4     106         128   
top              NaN             No       Male      40         USA   
freq             NaN           2038       1467      92         699   
mean    9.028016e+07            NaN        NaN     NaN         NaN   
std     7.435532e+03            NaN        NaN     NaN         NaN   
min     9.025877e+07            NaN        NaN     NaN         NaN   
25%     9.027520e+07            NaN        NaN     NaN         NaN   
50%     9.027780e+07            NaN        NaN     NaN         NaN   
75%     9.028242e+07            NaN        NaN     NaN         NaN   
max     9.031480e+07            NaN        NaN     NaN         NaN   

       Q5: STATE, PROVINCE, COUNTY, ETC Q6 | 100 Grand Bar  \
count                              2360               1713   
unique                             

# I chose these methods:

Chapter 7
* Transform data using mapping or a function
* Replace values

Chapter 8
* Create hierarchical index
* Pivot the data

Chapter 10
* Grouping with Dicts/Series
* Cross Tabs

Chapter 11
* Convert between string and date time
* Generate date range

## Chapter 7 - Transform data using mapping or a function

In [69]:
# Create a function for mapping the data. I chose changing gender values to a more standardized format.
def clean_gender(gender):
    if pd.isnull(gender):
        return np.nan
    gender = str(gender).strip().lower()
    if gender in ['male', 'm']:
        return 'Male'
    elif gender in ['female', 'f']:
        return 'Female'
    else:
        return 'Other'

# Apply the function to the "Q2: GENDER" column
candy_df['Q2: GENDER_clean'] = candy_df['Q2: GENDER'].map(clean_gender)

# Display value counts of the cleaned gender column
print(candy_df['Q2: GENDER_clean'].value_counts(dropna=False))

Q2: GENDER_clean
Male      1467
Female     839
Other      113
NaN         41
Name: count, dtype: int64


## Chapter 7 - Replace values

In [70]:
# Analyze the distribution of 'Q3: AGE'
age = pd.to_numeric(candy_df['Q3: AGE'], errors='coerce')
mean_age = age.mean()
median_age = age.median()
skewness = age.skew()

print(f"Mean: {mean_age:.2f}, Median: {median_age:.2f}, Skewness: {skewness:.2f}")

# This is a general function that I might want later. Skewness should determine whether or not we use mean or median.

# Decide replacement strategy
if abs(skewness) > 0.5:
    replacement = median_age
    print("Distribution is skewed. Replacing missing/invalid ages with median.")
else:
    replacement = mean_age
    print("Distribution is approximately normal. Replacing missing/invalid ages with mean.")

# Replace missing or invalid ages
candy_df['Q3: AGE_clean'] = age.fillna(replacement)

Mean: 42.61, Median: 41.00, Skewness: 28.20
Distribution is skewed. Replacing missing/invalid ages with median.


In [71]:
# I was going to transform these values, but the data is too messy.
# Instead, I will just print the unique values for context.
print(candy_df['Q4: COUNTRY'].unique())

[nan 'USA ' 'USA' 'us' 'usa' 'canada' 'Canada' 'Us' 'US' 'Murica'
 'United States' 'uk' 'United Kingdom' 'united states' 'Usa'
 'United States ' 'United staes' 'United States of America' 'UAE'
 'England' 'UK' 'canada ' 'Mexico' 'United states' 'u.s.a.' 'USAUSAUSA'
 'america' '35' 'france' 'United States of America ' 'U.S.A.' 'finland'
 'unhinged states' 'Canada ' 'united states of america' 'US of A'
 'Unites States' 'The United States' 'North Carolina ' 'Unied States'
 'Netherlands' 'germany' 'Europe' 'Earth' 'U S' 'u.s.' 'U.K. '
 'Costa Rica' 'The United States of America' 'unite states' 'U.S.' '46'
 'cascadia' 'Australia' 'insanity lately' 'Greece'
 'USA? Hard to tell anymore..' "'merica" 'usas' 'Pittsburgh' '45'
 'United State' '32' 'France' 'australia' 'A' 'Can' 'Canae' 'New York'
 'Trumpistan' 'Ireland' 'United Sates' 'Korea' 'California' 'Japan' 'USa'
 'South africa'
 'I pretend to be from Canada, but I am really from the United States.'
 'Usa ' 'Uk' 'Iceland' 'Germany' 'Canada`'

## Chapter 8 - Create hierarchical index


In [72]:
# I'm not sure which columns to use for the index, so I chose Country and Gender
candy_df_hier = candy_df.set_index(['Q4: COUNTRY', 'Q2: GENDER'])


# Using a hierarchical index (MultiIndex) in pandas allows rows or columns 
# to have multiple levels of labels, like a tree structure. 
# This helps organize and access data using more than one key.

# Another way of thinking of this is a "composite key"

# Display the first few rows to show the new index
print(candy_df_hier.head())

                        Internal ID Q1: GOING OUT? Q3: AGE  \
Q4: COUNTRY Q2: GENDER                                       
NaN         NaN            90258773            NaN     NaN   
USA         Male           90272821             No      44   
USA         Male           90272829            NaN      49   
us          Male           90272840             No      40   
usa         Male           90272841             No      23   

                       Q5: STATE, PROVINCE, COUNTY, ETC Q6 | 100 Grand Bar  \
Q4: COUNTRY Q2: GENDER                                                       
NaN         NaN                                     NaN                NaN   
USA         Male                                     NM                MEH   
USA         Male                               Virginia                NaN   
us          Male                                     or                MEH   
usa         Male                               exton pa                JOY   

                  

In [73]:
print(candy_df_hier.loc[('USA', 'Female')])

                        Internal ID Q1: GOING OUT? Q3: AGE  \
Q4: COUNTRY Q2: GENDER                                       
USA         Female         90272976             No      46   
            Female         90272978             No      50   
            Female         90272984             No      62   
            Female         90272994             No      62   
            Female         90273017             No      32   
...                             ...            ...     ...   
            Female         90313039             No      26   
            Female         90313279            NaN      67   
            Female         90313345             No      43   
            Female         90314022             No      26   
            Female         90314580             No      33   

                       Q5: STATE, PROVINCE, COUNTY, ETC Q6 | 100 Grand Bar  \
Q4: COUNTRY Q2: GENDER                                                       
USA         Female                   

C:\Users\Gabe\AppData\Local\Temp\ipykernel_6036\1561390008.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  print(candy_df_hier.loc[('USA', 'Female')])


## Chapter 8 - Pivot the data

In [74]:
# Pivot the data to see the average age by country and gender
pivot_table = candy_df.pivot_table(
    values='Q3: AGE_clean',
    index='Q4: COUNTRY',
    columns='Q2: GENDER_clean',
    aggfunc='mean'
)

# A pivot table summarizes data by grouping (rows) and aggregating (e.g., mean, sum) values, 
# optionally spreading categories across columns for comparison. Useful for reshaping and analyzing patterns.

print(pivot_table)

Q2: GENDER_clean             Female       Male      Other
Q4: COUNTRY                                              
'merica                         NaN   9.000000        NaN
1                               NaN        NaN   1.000000
32                              NaN  41.000000        NaN
35                        41.000000        NaN        NaN
45                        41.000000        NaN        NaN
...                             ...        ...        ...
united states of america  45.500000  40.666667        NaN
united ststes             44.000000        NaN        NaN
us                        45.944444  45.000000  40.333333
usa                       42.814815  45.057692  41.500000
usas                            NaN        NaN  67.000000

[128 rows x 3 columns]


## Chapter 10 - Grouping with Dicts/Series

In [75]:
# Group ages by cleaned gender using a dictionary and aggregate mean and count
gender_map = {'Male': 'M', 'Female': 'F', 'Other': 'O'}
# Map the cleaned gender to short codes
candy_df['Gender_Encoded'] = candy_df['Q2: GENDER_clean'].map(gender_map)

# Group by the new Gender_Encoded column and aggregate
gender_grouped = candy_df.groupby('Gender_Encoded')['Q3: AGE_clean'].agg(['mean', 'count'])
print(gender_grouped)

                     mean  count
Gender_Encoded                  
F               40.679380    839
M               42.900068   1467
O               51.761062    113


## Chapter 10 - Cross Tabs

In [76]:
# We can create a cross-tabulation of country and cleaned gender
cross_tab = pd.crosstab(candy_df['Q4: COUNTRY'], candy_df['Q2: GENDER_clean'], margins=True)
print(cross_tab)

Q2: GENDER_clean  Female  Male  Other   All
Q4: COUNTRY                                
'merica                0     1      0     1
1                      0     0      1     1
32                     0     1      0     1
35                     1     0      0     1
45                     1     0      0     1
...                  ...   ...    ...   ...
united ststes          1     0      0     1
us                    18    19      3    40
usa                   54   156      6   216
usas                   0     0      1     1
All                  835  1449    103  2387

[129 rows x 4 columns]


## Chapter 11 - Convert between string and date time

In [77]:
# Convert the 'Q11: DAY' column to datetime
# We can coerce to NaT
candy_df['Q11: DAY_dt'] = pd.to_datetime(candy_df['Q11: DAY'], errors='coerce')
print(candy_df[['Q11: DAY', 'Q11: DAY_dt']].head())


  Q11: DAY Q11: DAY_dt
0      NaN         NaT
1   Sunday         NaT
2      NaN         NaT
3   Sunday         NaT
4   Friday         NaT


C:\Users\Gabe\AppData\Local\Temp\ipykernel_6036\3009419657.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  candy_df['Q11: DAY_dt'] = pd.to_datetime(candy_df['Q11: DAY'], errors='coerce')


## Chapter 11 - Generate date range

In [78]:
# Generate a date range for 2015
date_range = pd.date_range(start='2015-01-01', end='2015-12-31', freq='D')
print(date_range)

DatetimeIndex(['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04',
               '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08',
               '2015-01-09', '2015-01-10',
               ...
               '2015-12-22', '2015-12-23', '2015-12-24', '2015-12-25',
               '2015-12-26', '2015-12-27', '2015-12-28', '2015-12-29',
               '2015-12-30', '2015-12-31'],
              dtype='datetime64[ns]', length=365, freq='D')
